# torch `nn.Embedding`

## 사전학습된 임베딩을 사용하지 않는 경우

In [1]:
# 감성 분석 샘플 데이터
sentences = [
    'nice great best amazing',
    'stop lies',
    'pitiful nerd',
    'excellent work',
    'supreme quality',
    'bad',
    'highly respectable'
]
labels = [1, 0, 0, 1, 1, 0, 1]

In [2]:
# 토큰화
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [4]:
# 단어 사전/정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]
word_counts = Counter(tokens)
print(word_counts)

word_to_index = {word:index+2 for index,word in enumerate(tokens)}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1
word_to_index = dict(sorted(word_to_index.items(),key=lambda x:x[1]))
print(word_to_index)

vocab_size = len(word_to_index)
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [7]:
def texts_to_sequences(sentences,word_to_index):
    sequences = []

    for sent in sentences:
        sequence = []
        for token in sent:
            if token in word_to_index:
                sequence.append(word_to_index[token])
            else:
                sequence.append(word_to_index['<UNK>'])

        sequences.append(sequence)
    return sequences

sequences = texts_to_sequences(tokenized_sentences,word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [8]:
# 패딩
import numpy as np

def pad_squences(sequences,maxlen):
    padded_squences = np.zeros([len(sequences),maxlen],dtype=int)
    for index, seq in enumerate(sequences):
        padded_squences[index, : len(seq)] = seq[:maxlen]
    return padded_squences

padded_squences = pad_squences(sequences,maxlen=4)
padded_squences

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [9]:
# 모델 생성
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

class SimpleNet(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size):
        super().__init__()
        self.embadding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0           # 패딩 벡터는 학습하지 않음
        )
        self.rnn = nn.RNN(embedding_dim,hidden_size,batch_first=True)
        self.out = nn.Linear(hidden_size,1)

    def forward(self,x):
        embbedded = self.embadding(x)
        _,h_n = self.rnn(embbedded)
        out = self.out(h_n.squeeze(0))
        return out

embedding_dim=100
model = SimpleNet(vocab_size,embedding_dim,hidden_size=8)
model

SimpleNet(
  (embadding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 8, batch_first=True)
  (out): Linear(in_features=8, out_features=1, bias=True)
)

In [10]:
import pandas as pd

# 학습 전 임베딩 벡터
wv = model.embadding.weight.data
print(wv.shape)

# 특정 단어 벡터
vocab = word_to_index.keys()
pd.DataFrame(wv,index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,-1.581655,0.027303,0.472419,0.566637,0.898406,1.510826,-0.113371,-1.165141,-0.578317,0.225833,0.671401,1.096332,0.994787,-0.189515,-0.796552,0.562166,1.019600,0.465870,0.269813,-0.981374,-0.924359,-1.266563,-0.480797,-0.692102,-1.023116,0.503027,-1.521321,-0.288741,0.582263,-1.000825,0.897678,0.396584,0.346768,0.484175,-0.737197,-0.518474,0.799105,-0.931635,0.759455,-0.302635,...,-0.818745,1.430139,2.316562,1.231387,0.142396,1.126729,2.203846,0.866743,-1.494911,0.588551,-1.270629,2.509761,-1.110707,-0.261058,-0.166477,0.808479,-1.085060,1.295333,-0.595598,1.094272,-1.261778,-0.072968,-0.122372,0.941351,-0.900286,-0.291262,-0.113354,0.448977,1.133371,1.722680,0.000461,-0.838062,-0.204138,1.266455,1.187328,0.186948,-0.498035,-0.459283,-1.476712,0.989287
nice,1.102290,1.057616,1.981297,-0.588157,1.167145,-0.048873,0.478345,0.134623,0.033740,0.386879,1.458311,-0.271289,1.199823,-0.940427,0.351494,1.375776,-0.976013,-0.896606,-0.248184,0.322037,-0.330599,1.643174,0.439577,-1.776485,-2.226309,0.558682,1.176891,-0.214662,0.462974,-0.821842,0.570808,0.029896,0.462516,0.233432,1.240007,-1.863457,-0.458301,-1.506458,-1.748365,0.246386,...,-0.542025,0.271005,-1.260703,0.310673,-0.754529,1.542278,0.534869,-1.034302,-1.086478,-0.150603,0.554870,-0.801593,-0.868416,-0.552042,-1.528399,-0.937079,0.725092,-1.030894,1.636920,-0.181503,-2.317660,-1.017013,0.828230,-0.143655,-0.820394,0.698036,0.720468,0.608901,1.672157,1.247442,0.750246,0.068685,-0.323905,-1.099979,0.400944,-0.698361,-1.905635,0.276331,-1.208762,-1.469257
great,-1.117050,-0.378348,-0.194045,0.457131,0.637951,1.359050,0.226557,-0.838712,-0.940349,-0.492713,1.082829,-0.379211,-0.361508,-1.830155,-0.062327,-1.575598,-0.356336,0.523854,0.329109,0.767027,-0.509865,0.360442,-0.220934,-0.410147,0.295726,0.889985,1.137260,0.922023,0.840191,0.199199,-1.497935,-1.333181,-1.456777,1.859255,0.581657,-1.352204,0.408772,-2.109450,0.096283,-1.871535,...,-0.421212,0.319005,-0.167307,-0.346707,1.494488,-0.956288,0.843938,-0.116013,1.906463,-0.048010,1.339648,-0.146766,0.492899,0.433233,-0.104789,-1.039004,0.431013,-1.018926,-0.774447,-1.462881,1.744563,-0.711033,0.648336,-0.430496,0.152015,-1.516213,0.311410,1.751812,0.152689,0.670207,-1.185082,-0.159290,0.166839,-0.053257,0.403647,0.213675,-0.276483,-0.897019,-0.614904,-1.344293
best,0.220408,-0.766197,-0.715310,-0.366164,-1.495370,-0.048152,-0.847262,-0.529026,-0.560469,0.022942,-0.184316,-0.473804,-0.943941,0.326852,0.233744,1.251859,0.198063,1.328561,0.653536,-1.281383,0.225177,0.595766,0.168339,-0.952520,-0.174728,-0.307375,-0.094263,-0.115999,0.860893,-0.820424,0.353983,0.123102,0.138609,-1.440257,0.128358,-0.089770,0.723163,0.704007,0.765295,0.103949,...,-1.119917,1.393615,-0.297420,-0.375700,-0.193049,-0.664986,-0.660028,-0.244283,0.262385,-2.318282,-0.577554,-0.745133,-1.659336,0.172235,-0.023320,0.385519,-0.764323,0.109702,-1.921472,1.185430,-0.240891,0.904384,0.975245,0.097037,0.833564,0.906013,2.458450,1.501921,-2.458398,-0.461504,0.621665,0.877608,0.302400,-0.462374,-1.165166,-0.

In [ ]:
# 모델 학습
X = torch.tensor(padded_squences,dtype=torch.long)              # (batch_size,seq_len)
y= torch.tensor(labels,dtype=torch.float).unsqueeze(1)          # (batch_size,1)

print(f'{X.shape}, {y.shape}')

dataset = TensorDataset(X,y)
dataloader = DataLoader(dataset,batch_size=2,shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.005)

torch.Size([7, 4]), torch.Size([7, 1])


In [ ]:
for epoch in range(20):
    epoch_loss = 0
    for x_batch,y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output,y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'epoch {epoch+1} : Loss : {epoch_loss / len(dataloader)}')

epoch 1 : Loss : 0.7139550000429153
epoch 2 : Loss : 0.6345522105693817
epoch 3 : Loss : 0.659809410572052
epoch 4 : Loss : 0.6475090235471725
epoch 5 : Loss : 0.6304106563329697
epoch 6 : Loss : 0.5927340164780617
epoch 7 : Loss : 0.5614373832941055
epoch 8 : Loss : 0.5324572026729584
epoch 9 : Loss : 0.4734615609049797
epoch 10 : Loss : 0.456064835190773
epoch 11 : Loss : 0.41850627958774567
epoch 12 : Loss : 0.3739061877131462
epoch 13 : Loss : 0.33644118905067444
epoch 14 : Loss : 0.31425290554761887
epoch 15 : Loss : 0.2548166438937187
epoch 16 : Loss : 0.23968198150396347
epoch 17 : Loss : 0.20070943236351013
epoch 18 : Loss : 0.17549549043178558
epoch 19 : Loss : 0.15937422960996628
epoch 20 : Loss : 0.13613549806177616
epoch 21 : Loss : 0.12280835025012493
epoch 22 : Loss : 0.10608835145831108
epoch 23 : Loss : 0.09482093900442123
epoch 24 : Loss : 0.08543062582612038
epoch 25 : Loss : 0.07652867212891579
epoch 26 : Loss : 0.06947429291903973
epoch 27 : Loss : 0.063846657983958

In [ ]:
# 평가/ 예측
model.eval()
with torch.no_grad():
    output = model(X)
    prob = torch.sigmoid(output)
    pred = (prob >= 0.5).int()

print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


In [15]:
# 학습 후 임베딩 벡터
wv = model.embadding.weight.data
print(wv.shape)

# 특정 단어 벡터
vocab = word_to_index.keys()
pd.DataFrame(wv,index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,-1.581655,0.027303,0.472419,0.566637,0.898406,1.510826,-0.113371,-1.165141,-0.578317,0.225833,0.671401,1.096332,0.994787,-0.189515,-0.796552,0.562166,1.019600,0.465870,0.269813,-0.981374,-0.924359,-1.266563,-0.480797,-0.692102,-1.023116,0.503027,-1.521321,-0.288741,0.582263,-1.000825,0.897678,0.396584,0.346768,0.484175,-0.737197,-0.518474,0.799105,-0.931635,0.759455,-0.302635,...,-0.818745,1.430139,2.316562,1.231387,0.142396,1.126729,2.203846,0.866743,-1.494911,0.588551,-1.270629,2.509761,-1.110707,-0.261058,-0.166477,0.808479,-1.085060,1.295333,-0.595598,1.094272,-1.261778,-0.072968,-0.122372,0.941351,-0.900286,-0.291262,-0.113354,0.448977,1.133371,1.722680,0.000461,-0.838062,-0.204138,1.266455,1.187328,0.186948,-0.498035,-0.459283,-1.476712,0.989287
nice,1.084368,1.118087,2.044694,-0.572709,1.271484,-0.108929,0.471489,0.170910,0.004349,0.386990,1.492266,-0.361203,1.158017,-0.906083,0.398338,1.360427,-0.970509,-0.830147,-0.221646,0.300120,-0.310145,1.620416,0.510860,-1.773236,-2.241285,0.554750,1.187427,-0.223022,0.425571,-0.836339,0.579062,0.137976,0.425753,0.203225,1.208382,-1.921022,-0.500433,-1.553796,-1.811866,0.230681,...,-0.539118,0.248025,-1.294282,0.397024,-0.812714,1.553712,0.520221,-1.011912,-1.102646,-0.161906,0.522240,-0.815121,-0.921203,-0.570560,-1.522627,-0.894732,0.725674,-1.039868,1.610061,-0.249852,-2.269599,-1.015811,0.774085,-0.192952,-0.806860,0.747011,0.707730,0.554477,1.711666,1.199295,0.667968,0.072620,-0.347175,-1.070362,0.258527,-0.643836,-1.951522,0.222110,-1.216451,-1.451392
great,-1.092873,-0.342970,-0.157977,0.462226,0.677913,1.314727,0.253300,-0.793782,-1.109854,-0.502047,1.111754,-0.401941,-0.381874,-1.855489,-0.062907,-1.521627,-0.301088,0.530001,0.360672,0.719276,-0.528404,0.352073,-0.184675,-0.400028,0.294490,0.893721,1.170121,0.958334,0.830343,0.174281,-1.512258,-1.272668,-1.568136,1.793475,0.557931,-1.384261,0.395037,-2.130651,0.109223,-1.852046,...,-0.426595,0.322015,-0.160891,-0.301392,1.472363,-0.987451,0.852363,-0.120060,1.886489,-0.039706,1.253506,-0.220822,0.460489,0.417677,-0.118817,-1.027190,0.423074,-0.951367,-0.748250,-1.376408,1.736747,-0.707812,0.619814,-0.453597,0.188348,-1.496768,0.295026,1.717837,0.133123,0.646720,-1.284495,-0.132434,0.131259,-0.046616,0.366949,0.213802,-0.316995,-0.908768,-0.581416,-1.371029
best,0.252093,-0.804658,-0.753133,-0.399345,-1.539085,-0.013375,-0.810811,-0.557938,-0.547675,-0.010281,-0.222511,-0.430419,-0.907091,0.288827,0.194651,1.287214,0.170835,1.291895,0.542163,-1.246256,0.189777,0.520794,0.133237,-0.895725,-0.146305,-0.235667,-0.127924,-0.086047,0.921222,-0.786461,0.331608,0.085954,0.169659,-1.408261,0.162919,-0.053957,0.786624,0.739568,0.802915,0.136575,...,-1.155135,1.423960,-0.258300,-0.414186,-0.154394,-0.674673,-0.647093,-0.215938,0.281444,-2.292068,-0.550768,-0.729475,-1.623642,0.201408,-0.024851,0.347280,-0.720959,0.174113,-1.890039,1.220178,-0.279790,0.871583,1.009413,0.132768,0.802005,0.870124,2.496557,1.535398,-2.496511,-0.421705,0.650280,0.833862,0.268866,-0.393464,-1.129416,-0

## 사전학습된 임베딩을 사용하는 경우

https://code.google.com/archive/p/word2vec/

In [17]:
# 사전 학습된 임베딩 로드
from gensim.models import KeyedVectors

model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin',binary=True)
model_wv.vectors.shape

(3000000, 300)

In [20]:
print(model_wv.most_similar('man',topn=5))
print(model_wv.most_similar('woman',topn=5))
print(model_wv.most_similar('king',topn=5))
print(model_wv.most_similar('queen',topn=5))

[('woman', 0.7664012908935547), ('boy', 0.6824871301651001), ('teenager', 0.6586930155754089), ('teenage_girl', 0.6147903203964233), ('girl', 0.5921714305877686)]
[('man', 0.7664012908935547), ('girl', 0.7494640946388245), ('teenage_girl', 0.7336829304695129), ('teenager', 0.6317085027694702), ('lady', 0.6288785934448242)]
[('kings', 0.7138045430183411), ('queen', 0.6510956883430481), ('monarch', 0.6413194537162781), ('crown_prince', 0.6204220056533813), ('prince', 0.6159993410110474)]
[('queens', 0.739944338798523), ('princess', 0.7070532441139221), ('king', 0.6510956883430481), ('monarch', 0.6383602023124695), ('very_pampered_McElhatton', 0.6357026696205139)]


In [22]:
print(model_wv.most_similar(positive=['king','woman'],negative=['man'],topn=5))
print(model_wv.most_similar(positive=['paris','italy'],negative=['france'],topn=5))

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581)]
[('lohan', 0.5069674849510193), ('madrid', 0.481842964887619), ('heidi', 0.4799900949001312), ('real_madrid', 0.4753323495388031), ('florence', 0.4682057499885559)]


In [25]:
# 모델의 Embedding layer에 초기화 할 embedding_matrix
print(len(word_to_index))       # (17,300) 크기의 embedding_matrix 생성

embedding_matrix = np.zeros((len(word_to_index),model_wv.vectors.shape[1]))
print(embedding_matrix.shape)

17
(17, 300)


In [26]:
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None

for word,index in word_to_index.items():
    if index >=2:
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb

In [27]:
pd.DataFrame(embedding_matrix,index=word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,-0.123535,-0.222656,-0.018066,-0.253906,0.131836,0.085938,0.161133,0.110840,-0.110840,-0.085938,0.026733,0.345703,0.151367,-0.004150,0.104980,0.049072,-0.069824,0.086426,0.031982,-0.028442,-0.157227,0.118652,0.361328,0.001732,0.052979,-0.234375,0.117676,0.086426,-0.011230,0.259766,...,-0.081055,-0.066895,-0.318359,-0.084473,0.135742,0.062500,0.070801,-0.142578,-0.112793,0.014526,-0.066895,0.038818,0.194336,0.095215,0.113770,-0.124512,0.137695,-0.188477,-0.052246,0.158203,0.098633,-0.043701,-0.060547,0.216797,0.040771,-0.146484,-0.189453,-0.251953,-0.168945,-0.086426,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,-0.033447,-0.185547,0.041016,-0.089844,0.021729,0.069336,0.180664,0.222656,-0.100586,-0.069336,0.000104,0.160156,0.040771,0.073730,0.153320,0.067871,-0.103027,0.041748,0.042725,-0.110352,-0.066895,0.041992,0.250000,0.212891,0.159180,0.014465,-0.048828,0.013977,0.003555,0.209961,...,-0.068359,-0.139648,-0.159180,-0.017944,0.021240,0.073730,0.130859,-0.080566,0.029907,0.015564,-0.166016,0.150391,-0.006775,0.010132,0.114746,-0.148438,-0.045898,-0.139648,-0.173828,-0.042725,-0.058105,0.052246,-0.111328,0.084473,-0.025513,0.140625,-0.181641,0.017212,-0.137695,-0.014771,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,0.139648,-0.086914,0.257812,0.073730,-0.018921,0.125000,0.090820,-0.001556,-0.031982,-0.145508,0.047607,0.173828,-0.146484,0.006012,0.030273,0.040771,-0.066406,0.184570,0.097168,-0.104980,0.024902,0.056396,0.165039,0.090820,0.185547,0.225586,-0.039795,-0.167969,-0.069336,0.019653,...,-0.178711,0.120605,-0.035889,0.095703,0.152344,0.003998,-0.059082,-0.032471,-0.054199,-0.005493,-0.045654,-0.001526,-0.050293,0.255859,0.048340,-0.019409,-0.127930,-0.088379,-0.225586,0.087402,0.205078,0.085938,0.066406,0.108398,-0.191406,0.070312,-0.163086,-0.002472,0.020264,0.001701,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0

In [28]:
# 모델 생성
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

class SimpleNet(nn.Module):
    def __init__(self,vocab_size,embedding_dim,embedding_matrix,hidden_size):
        super().__init__()
        self.embadding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0           # 패딩 벡터는 학습하지 않음
        )
        # 사전 학습된 임베딩 벡터로 초기화
        self.embadding.weight = nn.Parameter(torch.tensor(embedding_matrix,dtype=torch.float))
        # 사전 학습된 임베딩 벡터는 학습에 사용하지 않음
        self.embadding.weight.requires_grad = False

        self.rnn = nn.RNN(embedding_dim,hidden_size,batch_first=True)
        self.out = nn.Linear(hidden_size,1)

    def forward(self,x):
        embbedded = self.embadding(x)
        _,h_n = self.rnn(embbedded)
        out = self.out(h_n.squeeze(0))
        return out

embedding_dim=model_wv.vectors.shape[1]
model = SimpleNet(vocab_size,embedding_dim,embedding_matrix,hidden_size=8)
model

SimpleNet(
  (embadding): Embedding(17, 300, padding_idx=0)
  (rnn): RNN(300, 8, batch_first=True)
  (out): Linear(in_features=8, out_features=1, bias=True)
)

In [29]:
# 모델 학습
X = torch.tensor(padded_squences,dtype=torch.long)              # (batch_size,seq_len)
y= torch.tensor(labels,dtype=torch.float).unsqueeze(1)          # (batch_size,1)

print(f'{X.shape}, {y.shape}')

dataset = TensorDataset(X,y)
dataloader = DataLoader(dataset,batch_size=2,shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.005)

torch.Size([7, 4]), torch.Size([7, 1])


In [30]:
for epoch in range(20):
    epoch_loss = 0
    for x_batch,y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output,y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'epoch {epoch+1} : Loss : {epoch_loss / len(dataloader)}')

epoch 1 : Loss : 0.6889089047908783
epoch 2 : Loss : 0.6427295804023743
epoch 3 : Loss : 0.6122049540281296
epoch 4 : Loss : 0.5217395722866058
epoch 5 : Loss : 0.5076186954975128
epoch 6 : Loss : 0.47869356721639633
epoch 7 : Loss : 0.4281734451651573
epoch 8 : Loss : 0.3724142014980316
epoch 9 : Loss : 0.32294514775276184
epoch 10 : Loss : 0.27368783578276634
epoch 11 : Loss : 0.22431736439466476
epoch 12 : Loss : 0.18332578241825104
epoch 13 : Loss : 0.1529909111559391
epoch 14 : Loss : 0.12731528282165527
epoch 15 : Loss : 0.10853824950754642
epoch 16 : Loss : 0.09286828152835369
epoch 17 : Loss : 0.08066901378333569
epoch 18 : Loss : 0.06966807506978512
epoch 19 : Loss : 0.0618293983861804
epoch 20 : Loss : 0.05580116808414459


In [31]:
# 평가/ 예측
model.eval()
with torch.no_grad():
    output = model(X)
    prob = torch.sigmoid(output)
    pred = (prob >= 0.5).int()

print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]
